In [ ]:
# install dependencies (Colab-compatible)
import subprocess, sys

packages = [
    "datasets",
    "torch",
    "numpy",
    "matplotlib",
    "scikit-learn",  
    "tqdm",          
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All dependencies installed.")

All dependencies installed.


In [2]:
from datasets import load_dataset
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import re

RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

In [7]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

dataset = load_dataset("ag_news")

train_full = dataset["train"]
test = dataset["test"]

split = train_full.train_test_split(test_size = 0.10, seed = RANDOM_SEED, shuffle = True)

train = split["train"]
val = split["test"]

train_texts = train["text"]
train_labels = train["label"]

val_texts = val["text"]
val_labels = val["label"]

test_texts = test["text"]
test_labels = test["label"]

print(f"Train size: {len(train)}")
print(f"Val size:   {len(val)}")
print(f"Test size:  {len(test)}")

Train size: 108000
Val size:   12000
Test size:  7600


In [ ]:
# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# config
class Config:
    def __init__(self):
        self.lowercase        = True
        self.keep_punctuation = True
        self.add_sos          = True
        self.add_eos          = True
        self.min_freq         = 3
        self.max_len          = 128

        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        self.sos_token = "<SOS>"
        self.eos_token = "<EOS>"

config = Config()

# tokenizer
def tokenize(text, config):
    if config.lowercase:
        text = text.lower()
    if config.keep_punctuation:
        return re.findall(r"\w+|[^\w\s]", text)
    else:
        return re.findall(r"\w+", text)

# vocabulary builder
def build_vocab(texts, config):
    word_counts = {}
    for text in texts:
        tokens = tokenize(text, config)
        for token in tokens:
            word_counts[token] = word_counts.get(token, 0) + 1

    vocab   = {config.pad_token: 0, config.unk_token: 1}
    next_id = 2

    if config.add_sos:
        vocab[config.sos_token] = next_id; next_id += 1
    if config.add_eos:
        vocab[config.eos_token] = next_id; next_id += 1

    for word, freq in word_counts.items():
        if freq >= config.min_freq and word not in vocab:
            vocab[word] = next_id
            next_id += 1

    return vocab

# encoder
def encode_text(text, vocab, config):
    tokens = tokenize(text, config)

    if config.add_sos:
        tokens = [config.sos_token] + tokens
    if config.add_eos:
        tokens = tokens + [config.eos_token]

    ids = [vocab.get(t, vocab[config.unk_token]) for t in tokens]
    ids = ids[:config.max_len]

    # if truncated, preserve <EOS> at the last position
    if config.add_eos and len(ids) == config.max_len:
        ids[-1] = vocab[config.eos_token]

    return ids

# decoder
def decode(ids, vocab, config):
    skip       = {vocab[config.pad_token], vocab[config.unk_token],
                  vocab[config.sos_token], vocab[config.eos_token]}
    id_to_word = {v: k for k, v in vocab.items()}
    return " ".join(id_to_word[i] for i in ids if i not in skip)

# build vocab and encode splits
vocab = build_vocab(train_texts, config)

train_encoded = [encode_text(text, vocab, config) for text in train_texts]
val_encoded   = [encode_text(text, vocab, config) for text in val_texts]
test_encoded  = [encode_text(text, vocab, config) for text in test_texts]

# vocab statistics
def oov_rate(encoded_texts, unk_id=1):
    total     = sum(len(seq) for seq in encoded_texts)
    unk_count = sum(seq.count(unk_id) for seq in encoded_texts)
    return unk_count / total

print(f"\nVocab size:    {len(vocab):,}")
print(f"Val OOV rate:  {oov_rate(val_encoded):.4f}")
print(f"Test OOV rate: {oov_rate(test_encoded):.4f}")

# sequence length distribution
import matplotlib.pyplot as plt
from collections import Counter

raw_lengths = [len(tokenize(t, config)) for t in train_texts]
truncated   = sum(1 for l in raw_lengths if l > config.max_len)
print(f"\nSequences truncated: {truncated:,}/{len(raw_lengths):,} "
      f"({100 * truncated / len(raw_lengths):.1f}%)")

plt.figure(figsize=(8, 4))
plt.hist(raw_lengths, bins=50, color="steelblue", edgecolor="black")
plt.axvline(config.max_len, color="red", linestyle="--", label=f"L_max = {config.max_len}")
plt.xlabel("Sequence Length (tokens)")
plt.ylabel("Count")
plt.title("Training Set Token Length Distribution")
plt.legend()
plt.tight_layout()
plt.savefig("length_distribution.png", dpi=150)
plt.show()

# class balance
label_names = ["World", "Sports", "Business", "Sci/Tech"]
print()
for split_name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    counts = Counter(labels)
    print(f"{split_name} label distribution:")
    for i, name in enumerate(label_names):
        print(f"  {name:10s}: {counts[i]:6,}  ({100 * counts[i] / len(labels):.1f}%)")
    print()

# sanity check
print("Example text:  ", train_texts[0])
print("Example tokens:", tokenize(train_texts[0], config)[:20])
print("Example ids:   ", train_encoded[0][:20])
print("Example decode:", decode(train_encoded[0], vocab, config))

Vocab size: 42455
Example text: American Aphrodite Brooklyn-native Yvette Jarvis is an 'every woman' in Greece: professional basketball player, model, TV and talk show star and Athens councilwoman.
Example tokens: ['american', 'aphrodite', 'brooklyn', '-', 'native', 'yvette', 'jarvis', 'is', 'an', "'", 'every', 'woman', "'", 'in', 'greece', ':', 'professional', 'basketball', 'player', ',']
Example ids: [2, 4, 1, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 12, 15, 16, 17, 18, 19, 20]


In [5]:
class NewsDataset(Dataset):

    def __init__(self, encoded_texts, labels):
        self.encoded_texts = encoded_texts
        self.labels = labels

    def __len__(self):
        return len(self.encoded_texts)

    def __getitem__(self, idx):
        return self.encoded_texts[idx], self.labels[idx]

def collate_fn(batch):
    
    sequences = []
    labels = []
    lengths = []

    for seq, label in batch:
        sequences.append(seq)
        labels.append(label)
        lengths.append(len(seq))

    max_batch_len = max(lengths)
    pad_id = vocab[config.pad_token]

    padded_sequences = []
    for seq in sequences:
        padded_seq = seq + [pad_id] * (max_batch_len - len(seq))
        padded_sequences.append(padded_seq)

    input_ids = torch.tensor(padded_sequences, dtype = torch.long)
    lengths = torch.tensor(lengths, dtype = torch.long)
    labels = torch.tensor(labels, dtype = torch.long)

    return input_ids, lengths, labels

train_dataset = NewsDataset(train_encoded, train_labels)
val_dataset = NewsDataset(val_encoded, val_labels)
test_dataset = NewsDataset(test_encoded, test_labels)

train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True, collate_fn = collate_fn)
val_loader = DataLoader(val_dataset, batch_size = 64, shuffle = False, collate_fn = collate_fn)
test_loader = DataLoader(test_dataset, batch_size = 64, shuffle = False, collate_fn = collate_fn)

# input_ids, lengths, labels = next(iter(loader))